# Semana 13 – Validação e Otimização de Modelos
**Objetivo:** Aplicar GridSearch e Cross-Validation em um modelo do Titanic.

**Instruções:**
1. Baixar o notebook exemplo do repositório
2. Executar o código completo
3. Testar as mudanças sugeridas no Slide 15
4. Responder no notebook:
* Quais foram os melhores hiperparâmetros encontrados?
* O modelo otimizado teve uma acurácia melhor que o modelo padrão?
Quanto?
* O que aconteceu quando você mudou o número de folds no Cross-
Validation?
* Você acha que o GridSearch compensa o tempo de processamento? Por quê?

5. Subir o notebook respondido na pasta da semana 13 do repositório

# 1 - Instalar e importar bibliotecas

In [47]:
!pip install -q kagglehub

import kagglehub
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 2- Baixar e carregar o dataset

In [48]:
path = kagglehub.dataset_download("yasserh/titanic-dataset")

df = pd.read_csv(f"{path}/Titanic-Dataset.csv")

df.head()

Using Colab cache for faster access to the 'titanic-dataset' dataset.


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# 3 - Conhecendo os dados

In [49]:
df.info()

df.describe()

df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


# 4 - Pré-processamento

In [50]:
# Valores nulos
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Variáveis categóricas
df["Sex"] = df["Sex"].map({"male":0, "female":1})
df["Embarked"] = df["Embarked"].map({"S":0, "C":1, "Q":2})

# Remover colunas
df.drop(["PassengerId","Name","Ticket","Cabin"], axis=1, inplace=True)

df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,0,22.0,1,0,7.2500,0
1,1,1,1,38.0,1,0,71.2833,1
2,1,3,1,26.0,0,0,7.9250,0
3,1,1,1,35.0,1,0,53.1000,0
4,0,3,0,35.0,0,0,8.0500,0


# 5 - Separando treino e teste

In [51]:
X = df.drop("Survived", axis=1)
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# 6 - Modelo padrão

In [52]:
modelo = RandomForestClassifier(random_state=42)

modelo.fit(X_train, y_train)

pred = modelo.predict(X_test)

acc_padrao = accuracy_score(y_test, pred)

print(f"Acurácia: {acc_padrao:.4f}")

print(classification_report(y_test, pred))

Acurácia: 0.8268
              precision    recall  f1-score   support

           0       0.84      0.88      0.86       105
           1       0.81      0.76      0.78        74

    accuracy                           0.83       179
   macro avg       0.82      0.82      0.82       179
weighted avg       0.83      0.83      0.83       179



# 7 - GridSearchCV

In [53]:
param_grid = {
    "n_estimators":[50,100,200],
    "max_depth":[None,5,10,20],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,2,4]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [None, 5, 10, 20],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [50, 100, 200]},
             scoring='accuracy')

# 8 - Resultados do GridSearch

In [54]:
print("Melhores parâmetros:")
print(grid.best_params_)

print("\nMelhor score:")
print(grid.best_score_)

Melhores parâmetros:
{'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 100}

Melhor score:
0.8356544863587118


# 9 - Modelo otimizado

In [55]:
melhor_modelo = grid.best_estimator_

pred_grid = melhor_modelo.predict(X_test)

acc_grid = accuracy_score(y_test, pred_grid)

print(f"Acurácia otimizada: {acc_grid:.4f}")

Acurácia otimizada: 0.8045


# 10 - Comparação

In [56]:
print(f"Modelo padrão: {acc_padrao:.4f}")
print(f"Modelo otimizado: {acc_grid:.4f}")
print(f"Diferença: {acc_grid-acc_padrao:.4f}")

Modelo padrão: 0.8268
Modelo otimizado: 0.8045
Diferença: -0.0223


# 11 - Testando outros folds

In [57]:
for cv in [3, 5, 10]:
    grid = GridSearchCV(
        RandomForestClassifier(random_state=42),
        param_grid,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    print(f"CV = {cv} | Score = {grid.best_score_:.4f}")

CV = 3 | Score = 0.8329
CV = 5 | Score = 0.8357
CV = 10 | Score = 0.8343


# Respostas

## 1. Quais foram os melhores hiperparâmetros encontrados?

Os melhores hiperparâmetros encontrados pelo **GridSearchCV** foram:

- **max_depth:** 5
- **min_samples_leaf:** 2
- **min_samples_split:** 10
- **n_estimators:** 100

O melhor score obtido durante a validação cruzada foi de **0.8357**.

## 2. O modelo otimizado teve uma acurácia melhor que o modelo padrão? Quanto?

Não. O modelo otimizado apresentou uma acurácia inferior à do modelo padrão.

- **Modelo padrão:** 0.8268
- **Modelo otimizado:** 0.8045

A diferença foi de **-0.0223**, ou seja, o modelo otimizado teve uma redução de aproximadamente **2,23 pontos percentuais** na acurácia em relação ao modelo padrão.

## 3. O que aconteceu quando você mudou o número de folds no Cross-Validation?

Ao alterar o número de folds, houve uma pequena variação no desempenho do modelo. Com **3 folds**, o melhor score foi **0.8329**; com **5 folds**, o melhor resultado foi **0.8357**; e com **10 folds**, o score foi **0.8343**.

Nesse experimento, o **CV = 5** apresentou o melhor desempenho. Além disso, foi possível observar que aumentar o número de folds tende a aumentar o tempo de processamento, pois o modelo é treinado e validado mais vezes. Apesar disso, utilizar mais folds geralmente proporciona uma avaliação mais confiável do modelo.

## 4. Você acha que o GridSearch compensa o tempo de processamento? Por quê?

Neste experimento, o GridSearchCV não melhorou a acurácia do modelo, já que o modelo otimizado apresentou desempenho inferior ao modelo padrão. Mesmo assim, considero que o GridSearchCV é uma ferramenta útil, pois permite encontrar automaticamente a melhor combinação de hiperparâmetros dentro das opções testadas, evitando ajustes manuais. Em problemas mais complexos ou com outros modelos, essa busca pode resultar em melhorias significativas no desempenho.